# **Genomic analysis of malaria vectors in The Gambia**

### **Pyrethroid target-site resistance**


- In this notebook, compute and visualise frequencies of non-synonymous SNPs within the Vgsc gene for the chosen cohorts.

### **Metabolic resistance**


- Further, we compute and visualise the frequencies of CNVs in known metabolic gene classes for the chosen cohorts.


In [ ]:
!pip install -q malariagen_data

In [ ]:
import malariagen_data
import plotly.express as px
import pandas as pd

In [ ]:
ag3 = malariagen_data.Ag3()
ag3

In [ ]:
gambia_sample_query = "country == 'Gambia, The' and sample_set == '1272-VO-GM-OPONDO-VMF00160'"

In [ ]:
# access metadata
df_samples_gambia = ag3.sample_metadata(sample_query=gambia_sample_query)
df_samples_gambia

### **Pyrethroid target-site resistance**

- Vgsc gene target-site resistance

In [ ]:
df_samples_gambia.pivot_table(
    index=["admin1_name", "admin2_name", "year"],
    columns="taxon",
    values="sample_id",
    aggfunc="count",
    fill_value=0,
)

In [ ]:
df_vgsc_freq_gambia = ag3.aa_allele_frequencies(
    transcript="AGAP004707-RD",
    sample_query=gambia_sample_query,
    cohorts="admin2_year",
)
df_vgsc_freq_gambia

In [ ]:
ag3.plot_frequencies_heatmap(
    df_vgsc_freq_gambia.query("max_af > 0.05")
)

In [ ]:
df_vgsc_freq_gambia_admin2_year = (
    df_samples_gambia
    .groupby("cohort_admin2_year")
    .aggregate({
        "latitude": "mean",
        "longitude": "mean",
        "sample_id": "count",
    })
    .query("sample_id >= 10")
    .sort_values("longitude")
)
df_vgsc_freq_gambia_admin2_year

In [ ]:
all_cohorts_admin2_year = df_vgsc_freq_gambia_admin2_year.index.tolist()
all_cohorts_admin2_year

In [ ]:
ag3.plot_frequencies_heatmap(
    df_vgsc_freq_gambia.query("max_af > 0.05")[["label"] + [f"frq_{c}" for c in all_cohorts_admin2_year]]
)

In [ ]:
frequency_columns = [
    col for col in df_vgsc_freq_gambia.columns
    if col.startswith("frq_")
]
frequency_columns

In [ ]:
vgsc_aa_effect_df = df_vgsc_freq_gambia[["effect", "impact"]]
vgsc_aa_effect_df

In [ ]:
ds_vgsc_aa_gambia = ag3.aa_allele_frequencies_advanced(
    transcript="AGAP004707-RD",
    area_by="admin1_name",
    period_by="year",
    sample_query=gambia_sample_query,
    variant_query="max_af > 0.05"
)
ds_vgsc_aa_gambia

In [ ]:
ag3.plot_frequencies_time_series(
    ds_vgsc_aa_gambia,
    height=1500)

In [ ]:
arab_cohorts_admin2_year_gambia = (
    df_samples_gambia
    .query("taxon == 'arabiensis'")
    .groupby("cohort_admin2_year")
    .aggregate({
        "latitude": "mean",
        "longitude": "mean",
        "sample_id": "count",
    })
    .query("sample_id >= 10")
    .sort_values("longitude")
)
arab_cohorts_admin2_year_gambia

In [ ]:
arab_cohorts_admin2_year = arab_cohorts_admin2_year_gambia.index.tolist()
arab_cohorts_admin2_year

In [ ]:
ag3.plot_frequencies_heatmap(
    df_vgsc_freq_gambia.query("max_af > 0.05")[["label"] + [f"frq_{c}" for c in arab_cohorts_admin2_year]]
)

#### **Pyrethroid target-site resistance in gambiae and coluzzii**

In [ ]:
df_gambcolu_cohorts_admin2_year = (
    df_samples_gambia
    .query("taxon != 'arabiensis'")
    .groupby("cohort_admin2_year")
    .aggregate({
        "latitude": "mean",
        "longitude": "mean",
        "sample_id": "count",
        "taxon": "first",
    })
    .query("sample_id >= 10")
    .sort_values("taxon")
)
df_gambcolu_cohorts_admin2_year

In [ ]:
gambcolu_cohorts_admin2_year = df_gambcolu_cohorts_admin2_year.index.tolist()
gambcolu_cohorts_admin2_year

In [ ]:
ag3.plot_frequencies_heatmap(
    df_vgsc_freq_gambia.query("max_af > 0.05")[["label"] + [f"frq_{c}" for c in gambcolu_cohorts_admin2_year]]
)

### **Metabolic Resistance**

#### **Cyp P450**

- CNV freq across different species

In [ ]:
keap1 = "2R:40,926,195-40,945,169"
cyp6aap_region = "2R:28,480,000-28,510,000"
cyp9k1_region = "X:15,240,000-15,250,000"
cyp6mz_region = "3R:6,924,000-6,980,000"
cohorts = "admin2_year"
sample_sets = ["AG1000G-GM-A", "AG1000G-GM-B", "AG1000G-GM-C", "1272-VO-GM-OPONDO-VMF00160"]

In [ ]:
gst_region = "3R:28,597,000-28,600,000"
ace1_region = "2R:3,300,000-4,000,000"
ace_region = "2R:3,483,099-3,497,400"

In [ ]:
df_samples_gambia['release'].unique()

In [ ]:
ag3.sample_sets(release=['3.0', '3.5'])

In [ ]:
cnv_freqs_df_gambia = ag3.gene_cnv_frequencies(
    sample_query=gambia_sample_query,
    region=[keap1,cyp6aap_region,cyp9k1_region,cyp6mz_region, gst_region, ace_region],
    cohorts=cohorts,
    sample_sets=sample_sets,
    drop_invariant=False,
    )
cnv_freqs_df_gambia

In [ ]:
df_cnv_gambia_cohorts = (
    df_samples_gambia
    .groupby("cohort_admin2_year")
    .aggregate({
        "latitude":"mean",
        "longitude":"mean",
        "sample_id":"count",
        "taxon":"first"
    })
    .query("sample_id>=10")
    .sort_values("longitude")
)
df_cnv_gambia_cohorts

In [ ]:
df_cnv_gm_cohorts = df_cnv_gambia_cohorts.index.tolist()
df_cnv_gm_cohorts

In [ ]:
# frq_GM-M_Lower-Fuladu-West_arab_2019
# frq_GM-M_Lower-Fuladu-West_gcx2_2012
# frq_GM-N_Central-Badibu_gcx1_2011
# frq_GM-U_Basse_arab_2019
# frq_GM-U_Jimara_arab_2019
# frq_GM-W_Jeshwang_arab_2019
# frq_GM-W_Kombo-Central_arab_2019

In [ ]:
df_cnv_gm_cohorts=[
    'GM-W_Jeshwang_arab_2019',
    'GM-W_Kombo-Central_arab_2019',
    'GM-N_Central-Badibu_gcx1_2011',
    'GM-M_Lower-Fuladu-West_colu_2012',
    'GM-M_Lower-Fuladu-West_arab_2019',
    'GM-U_Jimara_arab_2019',
    'GM-U_Basse_arab_2019',
]

In [ ]:
ag3.plot_frequencies_heatmap(cnv_freqs_df_gambia.query("max_af > 0.05")[["label"] + [f"frq_{c}" for c in df_cnv_gm_cohorts]],
    title="Gene CNV frequencies, Cyp6aa/p locus + Cyp9k1 + Cyp6mz + Gst",
    row_height=40,
    color_continuous_scale="BuPu"
)

In [ ]:
ag3.plot_frequencies_heatmap(cnv_freqs_df_gambia.query("max_af > 0.05 and cnv_type == 'amp'"),
    title="Gene CNV frequencies, Cyp6aa/p locus + Cyp9k1 + Cyp6mz +Gst",
    row_height=40,
    color_continuous_scale="BuPu"
)

In [ ]:
arab2_cohorts_admin2_year = [
    'GM-W_Jeshwang_arab_2019',
    'GM-W_Kombo-Central_arab_2019',
    'GM-M_Lower-Fuladu-West_arab_2019',
    'GM-U_Jimara_arab_2019',
    'GM-U_Basse_arab_2019',
]

In [ ]:
cnv_freqs_df_gambia_cp = cnv_freqs_df_gambia.copy()
cnv_freqs_df_gambia_cp

In [ ]:
cnv_freqs_df_gambia_cp['label'] = cnv_freqs_df_gambia_cp['label'].str.replace('AGAP003645 amp', 'AGAP003645 (Keap1) amp')
cnv_freqs_df_gambia_cp['label'] = cnv_freqs_df_gambia_cp['label'].str.replace('AGAP003645 del', 'AGAP003645 (Keap1) del')
cnv_freqs_df_gambia_cp

In [ ]:
ag3.plot_frequencies_heatmap(
    cnv_freqs_df_gambia_cp.query("max_af > 0.05")[["label"] + [f"frq_{c}" for c in arab2_cohorts_admin2_year]],
    title="Gene CNV frequencies, Keap1 + Cyp6aa/p + Cyp9k1 + Cyp6mz + Gste1/2",
    row_height=30,
    height=500,
    width=700,
    color_continuous_scale="Turbo"
)

### **COE locus**

In [ ]:
coe_region = "2L:37,233,000-37,330,000"
coe_region_extended = "2L:37,217,000-37,335,000"

In [ ]:
cnv_freqs_df_gambia_coe = ag3.gene_cnv_frequencies(
    sample_query=gambia_sample_query,
    region=coe_region,
    cohorts=cohorts,
    sample_sets=sample_sets,
    drop_invariant=False,
    )
cnv_freqs_df_gambia_coe

In [ ]:
ag3.plot_frequencies_heatmap(
    cnv_freqs_df_gambia_coe.query("max_af > 0.05")[["label"] + [f"frq_{c}" for c in arab2_cohorts_admin2_year]],
    title="Gene CNV frequencies, COE locus",
    row_height=30,
    height=500,
    width=700,
    color_continuous_scale="Turbo"
)

### **Rdl - GABA CNVs**

In [ ]:
rdl_region = "2L:25,363,652-25,434,556"

In [ ]:
cnv_freqs_df_gambia_rdl = ag3.gene_cnv_frequencies(
    sample_query=gambia_sample_query,
    region=rdl_region,
    cohorts=cohorts,
    sample_sets=sample_sets,
    drop_invariant=False,
    )
cnv_freqs_df_gambia_rdl

In [ ]:
ag3.plot_frequencies_heatmap(
    cnv_freqs_df_gambia_rdl.query("max_af > 0.05")[["label"] + [f"frq_{c}" for c in arab2_cohorts_admin2_year]],
    title="Gene CNV frequencies, COE locus",
    row_height=30,
    height=500,
    width=700,
    color_continuous_scale="Turbo"
)

In [ ]:
color_mapping_admin2 = {
    'Jarra East': '#8e8c84',
    'Lower Fuladu West': '#7f7f7f',
    'Upper Fuladu West': '#6b705c',
    'Niani': '#4a4e4d',
    'Niamina East': '#3b5c40',
    'Jimara': '#8e7d32',
    'Basse': '#a95b29',
    'Wuli East': '#b64d3b',
    'Tumana': '#8a4b39',
    'Niamina West': '#6f4e37',
    'Sandu': '#3e3a3a',
    'Jeshwang': '#a4c9e2',
    'Kombo Central': '#78aaff',
    'Lower Niumi': '#4d88ff',
    'Bakau': '#0077b6',
    'Illiasa': '#0096c7',
    'Foni Brefet': '#00b4d8'
}

#### **GSTs and Ace1**

In [ ]:
gst_region = "3R:28,597,000-28,600,000"
ace1_region = "2R:3,300,000-4,000,000"
ace_region = "2R:3,483,099-3,497,400"
cohorts = "admin2_year"
sample_sets = ["AG1000G-GM-A", "AG1000G-GM-B", "AG1000G-GM-C", "1272-VO-GM-OPONDO-VMF00160"]

In [ ]:
cnv_freqs_df_gambia2 = ag3.gene_cnv_frequencies(
    sample_query=gambia_sample_query,
    region=[gst_region, ace_region],
    cohorts=cohorts,
    sample_sets=sample_sets,
    drop_invariant=False,
    )
cnv_freqs_df_gambia2

In [ ]:
ag3.plot_frequencies_heatmap(
    cnv_freqs_df_gambia2.query("max_af > 0.05")[["label"] + [f"frq_{c}" for c in arab2_cohorts_admin2_year]],
    title="Gene CNV frequencies, Gst locus + Ace1",
    row_height=40,
    color_continuous_scale="BuPu"
)

### **SNP Frequencies at metabolic sites**

**Diplotype clustering**

In [ ]:
keap1 = "2R:40,926,195-40,945,169"
cyp6aap_region = "2R:28,480,000-28,510,000"
cyp9k1_region = "X:15,240,000-15,250,000"
cyp6mz_region = "3R:6,924,000-6,980,000"
coe_region = "2L:37,233,000-37,330,000"
coe_region_extended = "2L:37,217,000-37,335,000"

In [ ]:
ag3.plot_haplotype_clustering(
    region='3R:28,590,000-28,600,000',
    analysis='gamb_colu',
    sample_sets='3.0',
    sample_query= 'country in ["Ghana", "Burkina Faso", "Cote d\'Ivoire"]',
    color='country',
    symbol='taxon'
)

Will do the COE region here first

In [ ]:
coe_region = "2L:37,250,000-37,300,000"

In [ ]:
ag3.genome_features(coe_region).query("type == 'mRNA'")

In [ ]:
ag3.plot_diplotype_clustering_advanced(
    region=coe_region,
    sample_sets='1272-VO-GM-OPONDO-VMF00160',
    cnv_region=coe_region,
    cnv_max_coverage_variance=0.8

    )

In [ ]:
ag3.plot_diplotype_clustering_advanced(
    region=coe_region,
    sample_sets='1272-VO-GM-OPONDO-VMF00160',
    cnv_region=coe_region,
    cnv_max_coverage_variance=0.8,
    snp_transcript='AGAP006723-RA'
    )

In [ ]:
ag3.plot_diplotype_clustering_advanced(
    region=coe_region,
    sample_sets='1272-VO-GM-OPONDO-VMF00160',
    cnv_region=coe_region,
    cnv_max_coverage_variance=0.8,
    snp_transcript='AGAP006724-RA'
    )

In [ ]:
ag3.plot_diplotype_clustering_advanced(
    region=coe_region,
    sample_sets='1272-VO-GM-OPONDO-VMF00160',
    cnv_region=coe_region,
    cnv_max_coverage_variance=0.8,
    snp_transcript='AGAP006725-RA'
    )

In [ ]:
ag3.plot_diplotype_clustering_advanced(
    region=coe_region,
    sample_sets='1272-VO-GM-OPONDO-VMF00160',
    cnv_region=coe_region,
    cnv_max_coverage_variance=0.8,
    snp_transcript='AGAP006726-RA'
    )

In [ ]:
ag3.plot_diplotype_clustering_advanced(
    region=coe_region,
    sample_sets='1272-VO-GM-OPONDO-VMF00160',
    cnv_region=coe_region,
    cnv_max_coverage_variance=0.8,
    snp_transcript='AGAP006727-RA'
    )

In [ ]:
ag3.plot_diplotype_clustering_advanced(
    region=coe_region,
    sample_sets='1272-VO-GM-OPONDO-VMF00160',
    cnv_region=coe_region,
    cnv_max_coverage_variance=0.8,
    snp_transcript='AGAP006728-RA'
    )

In [ ]:
ag3.plot_diplotype_clustering_advanced(
    region=coe_region,
    sample_sets='1272-VO-GM-OPONDO-VMF00160',
    cnv_region=coe_region,
    cnv_max_coverage_variance=0.8,
    snp_transcript='AGAP006729-RA'
    )

In [ ]:
ag3.plot_diplotype_clustering_advanced(
    region="2L:2,350,000-2,431,000",
    sample_sets='1272-VO-GM-OPONDO-VMF00160',
    cnv_region="2L:2,350,000-2,431,000",
    cnv_max_coverage_variance=0.8,
    snp_transcript='AGAP004707-RD'
    )